# 03 · Join Sofascore + Capology — Italy Serie A 24/25

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2024/25 de Serie A italiana**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_italy_2425.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_italy_2425.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  599 jugadores | 116 columnas
Capology:   700 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   inter
   milan

En Capology pero no en Sofascore:
   ac milan
   inter milan


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [7]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'inter milan':'inter',
            'ac milan':'milan'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [8]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 543/599 (90.7%)
Sin emparejar: 56


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [9]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          10
Revisión media    (0.75 ≤ score < 0.90):   8
Revisión estricta (0.50 ≤ score < 0.75):   21
Revisión muy est. (score < 0.50):           17


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [10]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
12,Albert Guðmundsson,Fiorentina,albert gudmundsson,0.971
5,Enrico Delprato,Parma,enrico del prato,0.968
16,Łukasz Skorupski,Bologna,lukasz skorupski,0.968
3,Evan Ndicka,Roma,evan n dicka,0.957
15,Þórir Jóhann Helgason,Lecce,thorir johann helgason,0.952
29,Konan N'Dri,Lecce,konan ndri,0.952
24,Lior Kasa,Genoa,lior kassa,0.947
30,Christian Gytkjær,Venezia,christian gytkjaer,0.941
39,Dani Silva,Hellas Verona,daniel silva,0.909
8,Milan Đurić,Parma,milan djuric,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [11]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
13,Faustino Anjorin,Empoli,tino anjorin,0.857
40,Dailon Rocha Livramento,Hellas Verona,dailon livramento,0.850
17,Mikael Ellertsson,Venezia,mikael egill ellertsson,0.850
18,Michel Adopo,Cagliari,michel ndary adopo,0.800
25,Cheick Oumar Condé,Venezia,cheick conde,0.800
6,Yann Bisseck,Inter,yann aurel bisseck,0.800
42,Marcus Pedersen,Torino,marcus holmgren pedersen,0.769
45,Mathias Løvik,Parma,mathias fjrtoft lvik,0.750


In [12]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 8 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [13]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
4,Mario Gila Fuentes,Lazio,mario gila,0.714
54,Lorenzo Tosto,Empoli,lorenzo colombo,0.714
2,Emerson Royal,Milan,emerson,0.700
35,Jacopo Bacci,Empoli,jacopo fazzini,0.692
19,Tommaso Rubino,Fiorentina,tommaso martinelli,0.688
1,Frank Anguissa,Napoli,andre zambo anguissa,0.647
47,Davide Bartesaghi,Milan,davide calabria,0.625
55,Nicola Pintus,Cagliari,nicolas viola,0.615
0,Juan Musso,Atalanta,juan cuadrado,0.609
26,Ismael Konate,Empoli,emmanuel ekong,0.593


In [14]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mario gila fuentes',
                    'emerson royal',
                    'frank anguissa',
                    'manu kone'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 4


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [15]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
21,Alessio Cacciamani,Torino,alberto paleari,0.485
28,Sergiu Perciun,Torino,alberto paleari,0.483
53,Oliver Abildgaard,Como,alieu fadera,0.483
31,Walid Cheddira,Napoli,david neres,0.480
33,Luka Topalović,Inter,marko arnautovic,0.467
52,Aaron Ciammaglichella,Torino,ivan ilic,0.467
46,Nicholas Pierini,Venezia,alessio zerbin,0.467
41,Valentin Gendrey,Lecce,antonino gallo,0.467
36,Vanja Vlahović,Atalanta,rafael toloi,0.462
20,Lorenzo Anghelè,Juventus,nicolas gonzalez,0.452


In [16]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [17]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 565/599 (94.3%)
Sin salario:     34


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [18]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 34


,player,team,minutesPlayed,appearances,goals,assists
0,Juan Musso,Atalanta,90,1,0,0
1,Vanja Vlahović,Atalanta,57,3,0,0
2,Federico Cassa,Atalanta,23,2,0,0
3,Mitchel Bakker,Atalanta,21,1,0,0
4,Alberto Manzoni,Atalanta,14,1,0,0
5,Nicola Pintus,Cagliari,3,1,0,0
6,Oliver Abildgaard,Como,7,1,0,0
7,Ismael Konate,Empoli,136,12,0,1
8,Viktor Kovalenko,Empoli,78,6,0,0
9,Jacopo Bacci,Empoli,29,3,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [19]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Atalanta  —  SF sin salario:


,player,minutesPlayed
0,Alberto Manzoni,14
1,Federico Cassa,23
2,Juan Musso,90
3,Mitchel Bakker,21
4,Vanja Vlahović,57


  CG plantilla completa:


,player,player_norm
0,Ademola Lookman,ademola lookman
1,Ben Godfrey,ben godfrey
2,Berat Djimsiti,berat djimsiti
3,Brandon Soppy,brandon soppy
4,Charles De Ketelaere,charles de ketelaere
5,Daniel Maldini,daniel maldini
6,Davide Zappacosta,davide zappacosta
7,Éderson,ederson
8,El Bilal Touré,el bilal toure
9,Francesco Rossi,francesco rossi



  Cagliari  —  SF sin salario:


...

,player,player_norm
0,Alexis Sánchez,alexis sanchez
1,Arthur Atta,arthur atta
2,Axel Guessand,axel guessand
3,Brenner,brenner
4,Christian Kabasele,christian kabasele
5,Damián Pizarro,damian pizarro
6,Daniele Padelli,daniele padelli
7,David Pejičić,david pejicic
8,Edoardo Piana,edoardo piana
9,Enzo Ebosse,enzo ebosse



  Venezia  —  SF sin salario:


,player,minutesPlayed
0,Nicholas Pierini,82
1,Nunzio Lella,8


  CG plantilla completa:


,player,player_norm
0,Alessandro Marcandalli,alessandro marcandalli
1,Alessio Zerbin,alessio zerbin
2,Alfred Duncan,alfred duncan
3,Antonio Candela,antonio candela
4,Antonio Raimondo,antonio raimondo
5,Bjarki Steinn Bjarkason,bjarki steinn bjarkason
6,Bruno Bertinato,bruno bertinato
7,Cheick Condé,cheick conde
8,Christian Gytkjaer,christian gytkjaer
9,Daniel Fila,daniel fila


In [20]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [21]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 565/599 (94.3%)


In [22]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [23]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_italy_2425.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_italy_2425.csv
   Jugadores totales:  599
   Con salario:        565
   Sin salario (NaN):  34
   Columnas:           121
